# 1. Creating Table 

In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.dim_suburb AS

WITH distinct_arrears_suburbs AS (
    -- Get distinct suburb names from suburb arrears
    SELECT DISTINCT 
        UPPER(TRIM(suburb)) AS suburb_name
    FROM cpt_utility_catalog.silver.silver_suburb_arrears_cleaned
    WHERE suburb IS NOT NULL
),

dominant_wards AS (
    -- Match suburbs with service requests to get the primary ward (mode)
    SELECT 
        UPPER(TRIM(suburb)) AS suburb_name,
        CAST(ward AS INT) AS ward_number,
        ROW_NUMBER() OVER (
            PARTITION BY UPPER(TRIM(suburb)) 
            ORDER BY COUNT(*) DESC
        ) AS rank
    FROM cpt_utility_catalog.silver.silver_service_requests_cleaned
    WHERE suburb IS NOT NULL AND ward IS NOT NULL
    GROUP BY UPPER(TRIM(suburb)), CAST(ward AS INT)
),

mapped_subcouncils AS (
    -- Map ward numbers to subcouncils
    SELECT 
        a.suburb_name,
        COALESCE(w.ward_number, -1) AS ward_number,
        CASE 
            -- Subcouncils by ward number
            WHEN w.ward_number IN (23, 29, 32, 107) THEN 1
            WHEN w.ward_number IN (6, 7, 8, 101, 102, 111) THEN 2
            WHEN w.ward_number IN (4, 113, 55, 56, 104) THEN 3
            WHEN w.ward_number IN (25, 26, 27, 28) THEN 4
            WHEN w.ward_number IN (12, 13, 20, 22, 24, 106) THEN 5
            WHEN w.ward_number IN (2, 3, 9, 10) THEN 6
            WHEN w.ward_number IN (1, 5, 21, 70, 103, 105, 112) THEN 7
            WHEN w.ward_number IN (15, 83, 84, 85, 86, 100, 109) THEN 8
            WHEN w.ward_number IN (18, 87, 89, 90, 91, 93) THEN 9
            WHEN w.ward_number IN (94, 95, 96, 97, 98, 99) THEN 10
            WHEN w.ward_number IN (30, 44, 46, 47, 48, 49, 60) THEN 11
            WHEN w.ward_number IN (35, 76, 82, 92, 116) THEN 12
            WHEN w.ward_number IN (34, 36, 37, 38, 39, 40, 41, 80) THEN 13
            WHEN w.ward_number IN (11, 14, 16, 17, 19, 108, 114) THEN 14
            WHEN w.ward_number IN (31, 42, 50, 51, 52) THEN 15
            WHEN w.ward_number IN (53, 54, 57, 77, 115) THEN 16
            WHEN w.ward_number IN (33, 43, 75, 78, 79, 81, 88) THEN 17
            WHEN w.ward_number IN (65, 66, 67, 68, 72, 110) THEN 18
            WHEN w.ward_number IN (45, 61, 64, 69) THEN 19
            WHEN w.ward_number IN (58, 59, 62, 63, 71, 73, 74) THEN 20
            ELSE -1
        END AS subcouncil
    FROM distinct_arrears_suburbs a
    LEFT JOIN dominant_wards w 
           ON a.suburb_name = w.suburb_name 
          AND w.rank = 1
)

-- Map subcouncil to city region
SELECT 
    xxhash64(LOWER(suburb_name)) AS suburb_key,
    INITCAP(suburb_name) AS suburb_name,
    ward_number AS ward,
    subcouncil,
    CASE 
        WHEN subcouncil IN (1, 3) THEN 'Blouberg / West Coast'
        WHEN subcouncil IN (2, 4, 6, 7) THEN 'Northern Suburbs'
        WHEN subcouncil IN (5, 9, 10, 11, 12, 13, 14, 17, 23) THEN 'Cape Flats'
        WHEN subcouncil IN (8, 21, 22, 24) THEN 'Helderberg & Eastern'
        WHEN subcouncil IN (15, 18, 20) THEN 'Southern Suburbs'
        WHEN subcouncil = 16 THEN 'City Bowl & Atlantic Seaboard'
        WHEN subcouncil = 19 THEN 'Southern Peninsula'
        ELSE 'UNKNOWN'
    END AS city_region
FROM mapped_subcouncils;